# AI 시험 연습 문제

Time Series / NCF / NGCF 코드 빈칸 채우기

## Setup

In [ ]:
# [코드]
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 문제 1. Sliding Window

[문제]
Q 1) 1차원 시계열 데이터를 딥러닝 모델 학습을 위한 (입력 시퀀스, 다음 시점 타깃) 쌍으로 분할하는 함수를 완성합니다.

[요구사항]
- sequence_length 길이의 연속된 데이터 구간을 입력 특징 리스트에 추가
- 해당 구간 바로 다음 시점의 1개 값을 타깃 리스트에 추가

In [ ]:
# [코드]
sequence_length = 50

def convert_data_into_tensors(data_seq):
    features, labels = [], []
    for i in range(len(data_seq) - sequence_length):
        features.append(### 작성 필요 ###)
        labels.append(### 작성 필요 ###)
    features, labels = np.array(features), np.array(labels)

    features = torch.tensor(features, dtype=torch.float32)
    labels = torch.tensor(labels, dtype=torch.float32)
    return features, labels

## 문제 2. LSTM Model

[문제]
Q 2) 시계열 예측을 위한 LSTM 모델 클래스를 정의하고 순전파(forward) 로직을 완성합니다.

[요구사항]
- nn.LSTM 모듈을 사용하여 input_size, hidden_size, num_layers 및 batch_first=True 설정
- 은닉 상태를 단일 예측값으로 매핑하는 nn.Linear 출력층 연결
- forward에서 LSTM 출력 중 마지막 시점(last time step)의 은닉 벡터를 추출하여 선형층 통과

In [ ]:
# [코드]
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super(LSTMModel, self).__init__()
        self.lstm = ### 작성 필요 ###
        self.linear = ### 작성 필요 ###

    def forward(self, x):
        out, _ = ### 작성 필요 ###
        return ### 작성 필요 ###

## 문제 3. Training Loop

[문제]
Q 3) 시계열 회귀 모델의 1 에폭 학습 루프(순전파, 손실 계산, 역전파 갱신)를 완성합니다.

[요구사항]
- 배치 입력 x에 대한 모델 순전파 수행 및 loss_fn을 통한 MSE 손실 계산
- 기울기 초기화(zero_grad), 역전파(backward), 파라미터 갱신(step) 순차 수행

In [ ]:
# [코드]
def train_one_epoch(model, train_loader, loss_fn, optimizer):
    model.train()
    total_loss = 0.0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        pred = ### 작성 필요 ###
        loss = ### 작성 필요 ###

        ### 작성 필요 ###
        ### 작성 필요 ###
        ### 작성 필요 ###

        total_loss += loss.item()
    return total_loss / len(train_loader)

## 문제 4. Conv1D Model

[문제]
Q 4) 시계열 데이터를 1D 합성곱(Conv1D)으로 처리하기 위해 텐서 형상을 변경하고 합성곱을 통과시킵니다.

[요구사항]
- (batch, seq_len, features) 형태의 입력을 nn.Conv1d 규격인 (batch, in_channels, seq_len)으로 축 교환(transpose)
- 합성곱 레이어 통과 후 다시 원래의 축 순서로 복원

In [ ]:
# [코드]
class Conv1DModel(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(Conv1DModel, self).__init__()
        self.conv1d = nn.Conv1d(in_channels=input_size,
                                out_channels=hidden_size,
                                kernel_size=2, stride=1)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        x = ### 작성 필요 ###
        x = self.conv1d(x)
        x = ### 작성 필요 ###
        return self.fc(x)

## 문제 5. Encoder-Decoder Forward

[문제]
Q 5) Seq2Seq RNN 모델에서 인코더의 최종 은닉 상태를 디코더의 초기 은닉 상태로 전달하고 재귀적 순전파를 수행합니다.

[요구사항]
- 인코더에 입력 시퀀스를 전달하여 최종 문맥 은닉 상태 h 추출
- 타깃 길이만큼 루프를 돌며 디코더에 이전 예측값을 순차 입력하여 미래 시퀀스 생성
- ※ 사전 정의 컴포넌트: EncoderRNN(input_size, hidden_size, num_layers), DecoderRNN(input_size, hidden_size, num_layers) 가 제공됩니다.

In [ ]:
# [코드]
# [참고 모델 정의 - 세부 로직 비공개]
# ### 공개 제한 ###
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super(EncoderRNN, self).__init__()
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

    def forward(self, x):
        _, h = self.rnn(x)
        return h

class DecoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super(DecoderRNN, self).__init__()
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, input_size)

    def forward(self, x, h):
        out, h = self.rnn(x, h)
        out = self.fc(out)
        return out, h

class RNNRNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super(RNNRNN, self).__init__()
        self.encoder = EncoderRNN(input_size, hidden_size, num_layers)
        self.decoder = DecoderRNN(input_size, hidden_size, num_layers)

    def forward(self, source, target_len):
        h = ### 작성 필요 ###
        predictions = []
        input = ### 작성 필요 ###
        for t in range(target_len):
            out, h = self.decoder(input, h)
            predictions.append(out.squeeze(1))
            input = ### 작성 필요 ###
        outputs = ### 작성 필요 ###
        return outputs

## 문제 6. NCF Model

[문제]
Q 6) NCF(Neural Collaborative Filtering) 모델의 유저/아이템 임베딩 테이블을 정의하고 결합 임베딩을 생성합니다.

[요구사항]
- 유저 수와 아이템 수에 대응하는 nn.Embedding 레이어 생성
- 유저 임베딩 벡터와 아이템 임베딩 벡터를 마지막 차원 기준으로 결합(torch.cat)하여 MLP 입력 구성

In [ ]:
# [코드]
class Neural_Collaborative_Filtering(nn.Module):
    def __init__(self, n_users, n_movies):
        super().__init__()
        self.user_embedding = ### 작성 필요 ###
        self.movie_embedding = ### 작성 필요 ###

        self.fc1 = nn.Linear(64, 32)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(32, 1)

    def forward(self, users, movies, ratings=None):
        user_embedding = self.user_embedding(users)
        movie_embedding = self.movie_embedding(movies)
        input_embedding = ### 작성 필요 ###

        hidden_feature = self.fc1(input_embedding)
        hidden_feature = self.relu(hidden_feature)
        output = self.fc2(hidden_feature)
        return output

## 문제 7. Precision@k & Recall@k

[문제]
Q 7) 추천 시스템 평가 지표인 Precision@k 및 Recall@k를 계산하는 함수를 작성합니다.

[요구사항]
- 실제 관련 있는 아이템 수(n_real_relevant)와 추천된 상위 k개 중 실제 관련 있는 아이템 수(n_real_relevant_in_top_k) 집계
- Precision@k = (추천된 관련 아이템 수) / k, Recall@k = (추천된 관련 아이템 수) / (전체 관련 아이템 수) 계산

In [ ]:
# [코드]
def compute_precision_recall(user_est_true, k=10, threshold=3.5):
    precisions = dict()
    recalls = dict()
    for user_id, user_ratings in user_est_true.items():
        user_ratings.sort(key=lambda x: x[0], reverse=True)

        n_real_relevant = ### 작성 필요 ###
        recommended_k = ### 작성 필요 ###
        n_real_relevant_in_top_k = ### 작성 필요 ###

        precisions[user_id] = ### 작성 필요 ###
        if n_real_relevant:
            recalls[user_id] = ### 작성 필요 ###
        else:
            recalls[user_id] = 0
    return precisions, recalls

## 문제 8. NGCF Layer: Normalization

[문제]
Q 8) NGCF(Neural Graph Collaborative Filtering) 그래프 합성곱을 위한 대칭 라플라시안 정규화 계수를 계산합니다.

[요구사항]
- 차수 벡터 deg로부터 norm = 1 / sqrt(deg[src] * deg[dst]) 정규화 계수 계산

In [ ]:
# [코드]
def compute_norm(deg, src, dst):
    norm = ### 작성 필요 ###
    return norm

## 문제 9. NGCF Layer: Edge Messages

[문제]
Q 9) NGCF 레이어에서 이웃 노드로부터 전달되는 엣지 메시지를 선형 변환 및 요소별 곱으로 계산합니다.

[요구사항]
- 이웃 노드 피처에 대한 단일 선형 변환과 두 노드 상호작용 피처에 대한 선형 변환을 결합
- 노드 정규화 계수를 곱하여 소스 노드와 타깃 노드로 향하는 메시지 텐서 산출

In [ ]:
# [코드]
class NGCFLayer(nn.Module):
    def __init__(self, input_dim, output_dim, dropout=0.1):
        super().__init__()
        self.W1 = nn.Linear(input_dim, output_dim)
        self.W2 = nn.Linear(input_dim, output_dim)
        self.leaky_relu = nn.LeakyReLU(0.2)

    def forward(self, edge_index, node_features, user_num, item_num):
        src, dst = edge_index

        deg = torch.zeros(node_features.size(0), device=node_features.device)
        deg.index_add_(0, src, torch.ones_like(src, dtype=torch.float))
        deg.index_add_(0, dst, torch.ones_like(dst, dtype=torch.float))

        norm = 1.0 / torch.sqrt(deg[src] * deg[dst])

        src_feat = node_features[src]
        dst_feat = node_features[dst]

        edge_messages_for_src = ### 작성 필요 ###
        edge_messages_for_src *= norm.unsqueeze(1)

        edge_messages_for_dst = ### 작성 필요 ###
        edge_messages_for_dst *= norm.unsqueeze(1)

        aggregated_messages = torch.zeros_like(node_features)
        aggregated_messages.index_add_(0, src, edge_messages_for_src)
        aggregated_messages[:user_num] += self.W1(node_features[:user_num])
        aggregated_messages.index_add_(0, dst, edge_messages_for_dst)
        aggregated_messages[user_num:] += self.W1(node_features[user_num:])

        return self.leaky_relu(aggregated_messages)

## 문제 10. NGCF: Final Features

[문제]
Q 10) NGCF의 각 레이어 출력 임베딩들을 결합(concatenate)하여 최종 유저 및 아이템 표현 벡터를 생성합니다.

[요구사항]
- 0차부터 L차까지의 모든 레이어 출력 텐서들을 마지막 특징 차원(dim=-1) 기준으로 결합

In [ ]:
# [코드]
def build_final_features(layer_outputs, num_users):
    final_features = ### 작성 필요 ###
    user_features = final_features[:num_users]
    item_features = final_features[num_users:]
    return user_features, item_features

## 문제 11. BPR Loss

[문제]
Q 11) BPR(Bayesian Personalized Ranking) 목적식을 바탕으로 선호 아이템과 비선호 아이템 간의 순위 손실을 계산합니다.

[요구사항]
- 유저 임베딩과 양성/음성 아이템 임베딩의 내적으로 각각 선호 점수 산출
- 점수 차이에 대해 -F.logsigmoid를 적용하고 정규화 항(reg_weight)을 더한 최종 손실 반환

In [ ]:
# [코드]
def bpr_loss(user_emb, pos_item_emb, neg_item_emb, reg_weight=1e-4):
    pos_scores = ### 작성 필요 ###
    neg_scores = ### 작성 필요 ###
    loss = ### 작성 필요 ###
    reg_loss = reg_weight * (user_emb.norm(2).pow(2) + pos_item_emb.norm(2).pow(2) + neg_item_emb.norm(2).pow(2)) / user_emb.size(0)
    return loss + reg_loss